# 과제 1. 도로망 최단경로와 시간대별 통행시간 분석

하남시 도로망을 이용하여 자유류 속도와 오후 첨두 속도로 계산한 최단경로와 통행시간을 비교하시오.
계산 결과를 바탕으로 통행시간의 차이가 차량의 다음 배차 가능 시각에 미치는 영향을 분석하시오.

범위는 교재 1~4장이며, 개인 과제입니다. 예상 소요 시간은 2~3시간입니다.
데이터 로딩, 분석 대상 지점 설정, 표·그림 출력 코드는 제공됩니다.
코드의 빈칸을 완성하고, 각 문제의 서술 답안을 작성하여 제출하시오.
하남시 경로 계산에는 제공된 `dijkstra` 함수를 사용하시오.

| 문제 | 내용 | 제출할 결과 | 배점 |
|---|---|---|---|
| 1 | 도로망 데이터와 엣지 통행시간 | 데이터 요약, 통행시간 계산, 서술 답안 | 20 |
| 2 | 다익스트라 알고리즘의 탐색 과정 | 단계별 표, 최단경로와 비용, 서술 답안 | 25 |
| 3 | 자유류와 오후 첨두의 경로·통행시간 비교 | 구간별 비교표, 경로 그림, 서술 답안 | 35 |
| 4 | 차량의 다음 배차 가능 시각 | 차량 일정표, 결과 해석 | 20 |

참고: [1장](../ko/ch01_why.md), [2장](../ko/ch02_road_network.md),
[3장](../ko/ch03_dijkstra.md), [4장](../ko/ch04_speeds_engine.md).

학번:  
이름:

## 준비 — 데이터와 도구 불러오기

이 노트북은 교재 저장소의 `assignment/` 폴더에서 엽니다.
교재의 `smartmob/` 코드와 `data/hanam/` 데이터가 함께 필요합니다.
노트북 파일만 받았다면 교재 저장소도 내려받아 이 파일을 `assignment/`에 넣습니다.

0장의 가상환경과 패키지 설치를 마쳤다면, 저장소 폴더의 터미널에서 다음 명령으로 엽니다.

```bash
jupyter lab assignment/assignment_1.ipynb
```

패키지가 없다는 오류가 나면 0장에서 만든 가상환경을 활성화하고
`python -m pip install -r requirements.txt`를 실행합니다.
주피터에서도 그 가상환경의 파이썬 커널을 선택합니다.
환경 준비를 마친 뒤에는 외부 서버나 API 키 없이 계산할 수 있습니다.

아래 코드를 실행하면 저장소 위치를 찾고, 분석에 사용할 도구를 불러옵니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "smartmob").is_dir() and (folder / "data" / "hanam").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "교재 저장소를 찾지 못했습니다. smartmob/와 data/가 있는 저장소의 "
        "assignment/ 폴더에 이 노트북을 놓고 다시 엽니다."
    )
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from smartmob.data import load_road_graph
from smartmob.teaching.dijkstra import dijkstra
from smartmob.teaching.graph import haversine_km
from smartmob.viz import use_korean_font
from lab import expect, todo

use_korean_font()
print("교재 저장소:", ROOT)

### 제공 데이터와 단위

2~4장에서 사용한 하남시 도로망을 그대로 사용합니다.
도로 연결은 OpenStreetMap 기반이며, 속도는 교재 데이터에 들어 있는 값을 사용합니다.
아래 두 파일은 이미 저장소에 있으므로 새로 수집할 필요가 없습니다.
`parquet`은 행과 열로 된 표를 저장하는 파일 형식입니다.

| 파일 | 내용 |
|---|---|
| `data/hanam/road_graph_nodes.parquet` | 도로망 노드의 식별자와 좌표 |
| `data/hanam/road_graph_edges.parquet` | 방향이 있는 도로 구간과 길이·속도 |

| 컬럼 | 의미와 단위 |
|---|---|
| `node_id` | 노드 식별자 |
| `lat`, `lon` | 위도, 경도. 함수에 넣을 때도 `(위도, 경도)` 순서 |
| `edge_id` | 엣지 식별자. 끝의 두 숫자에 출발·도착 노드 번호가 포함됨 |
| `highway` | 도로 종류. 자동차용 도로를 고르는 데 사용 |
| `length` | 엣지 길이, 미터(m) |
| `oneway` | 원래 도로의 일방통행 여부. 그래프에서는 각 엣지가 방향을 가짐 |
| `free_flow_speed_kmh` | 교통량의 방해가 적은 상태를 가정한 자유류 속도, km/h |
| `weekday_pm_peak_p50` | 주중 17~19시 관측 속도의 중앙값, km/h |

`load_road_graph(..., modes=("drive",))`는 자동차용 도로를 골라 방향 그래프를 만듭니다.
엣지 통행시간은 `length / 속도 * 3.6`으로 계산하며, 단위는 초입니다.
오후 첨두 속도가 비어 있으면 같은 엣지의 자유류 속도를 쓰고, 그것도 없으면 30km/h를 씁니다.
이 대체 규칙은 함수에 구현되어 있습니다. `edges`의 원래 결측값은 그대로 남아 있어 확인할 수 있습니다.

두 그래프의 도로 연결과 길이는 같고, 통행시간 계산에 쓰는 속도 컬럼이 다릅니다.
자유류는 기준 가정이며, 특정 날짜나 시각의 실제 주행을 뜻하지 않습니다.
각 경로를 계산하는 동안에는 선택한 속도 조건을 고정합니다.

In [ ]:
DATA_DIR = ROOT / "data" / "hanam"
nodes = pd.read_parquet(
    DATA_DIR / "road_graph_nodes.parquet",
    columns=["node_id", "lat", "lon"],
)
g_free = load_road_graph(
    "hanam", modes=("drive",), speed_column="free_flow_speed_kmh"
)
g_peak = load_road_graph(
    "hanam", modes=("drive",), speed_column="weekday_pm_peak_p50"
)
graphs = {"자유류": g_free, "오후 첨두": g_peak}
edges = g_free.edges  # 전체 원본에서 자동차용으로 고른 엣지 표입니다.

display(nodes.head())
display(edges[[
    "edge_id", "highway", "length", "oneway",
    "free_flow_speed_kmh", "weekday_pm_peak_p50",
]].head())

첫 표는 노드와 좌표, 둘째 표는 자동차용 엣지를 보여 줍니다.
`NaN`은 관측값이 없다는 뜻입니다. 속도가 0이라는 뜻이 아닙니다.
`nodes`에는 다른 통행수단의 노드도 있으므로 자동차 그래프의 노드 수는 `g_free.n_nodes`로 확인합니다.

### 비교할 세 구간

A와 B는 교재에서 쓴 하남시청·미사역의 대표 좌표입니다.
C는 A에서 위도를 0.01도 늘려 만든 가상 승하차 지점입니다.
아래 코드가 지점 표 `places`와 출발지·목적지 표 `trips`를 만듭니다.
세 통행은 과제용 사례이며 실제 호출 기록이 아닙니다.
각 구간을 독립적으로 비교하며, 한 차량이 세 구간을 연속해서 달린다는 뜻도 아닙니다.

A→B와 B→A를 따로 넣었습니다. 방향이 있는 도로망에서 두 방향의 결과를 살펴봅니다.

In [ ]:
places = pd.DataFrame([
    ["A", "하남시청", 37.5393, 127.2148],
    ["B", "미사역", 37.5606, 127.1930],
    ["C", "가상 승하차 지점", 37.5493, 127.2148],
], columns=["지점", "이름", "위도", "경도"]).set_index("지점")

trips = pd.DataFrame([
    ["A→B", "A", "B"],
    ["B→A", "B", "A"],
    ["C→B", "C", "B"],
], columns=["구간", "출발", "도착"])

# 같은 지점은 두 속도 조건에서 같은 노드에 연결합니다.
node_ids = {}
for key, row in places.iterrows():
    node_ids[key] = g_free.nearest_node(row["위도"], row["경도"])
    assert node_ids[key] in g_peak.nodes

places["노드"] = pd.Series(node_ids)
places["노드까지_m"] = [
    haversine_km(row["위도"], row["경도"], *g_free.coord[node_ids[key]]) * 1000
    for key, row in places.iterrows()
]
display(places.round(4))
display(trips)

`노드까지_m`은 입력 좌표와 계산에 사용할 노드 사이의 거리입니다.
과제의 통행시간에는 이 간격을 이동하는 시간이 포함되지 않습니다.

뒤에서 사용할 함수와 결과 형식은 다음과 같습니다.

| 코드 | 의미 |
|---|---|
| `graph.n_nodes`, `graph.n_edges` | 그래프의 노드 수, 방향이 있는 엣지 수 |
| `graph.nearest_node(위도, 경도)` | 좌표에 가까운 그래프 노드 선택 |
| `graph.neighbors(u)` | 노드 `u`에서 갈 수 있는 `(다음 노드, 통행시간_초, 엣지 행 번호)` 목록 |
| `dijkstra(graph, source, target)` | 노드 식별자 두 개로 최소 통행시간 경로 계산 |
| `path.duration_s` | 계산한 경로의 통행시간, 초 |
| `path.nodes` | 경로에 포함된 노드의 순서 |
| `path.coords(graph)` | 경로를 `(위도, 경도)` 좌표 목록으로 변환 |

여기서 가져온 `dijkstra`는 결과 객체를 반환합니다.
3장의 빈칸 파일 `labs/ch03_dijkstra.py`에 작성한 함수와 반환 형식이 다릅니다.
이 과제에서는 위 표의 호출 형식을 사용합니다.

## 문제 1. 도로망 데이터와 엣지 통행시간 (20점)

제공된 자동차 도로망 `g_free`와 엣지 표 `edges`를 이용하여 다음에 답하시오.

1. 자동차 도로망의 노드 수와 엣지 수를 구하시오.
2. 전체 자동차용 엣지 중 오후 첨두 속도(`weekday_pm_peak_p50`)가 결측인 엣지의 비율을 구하시오.
   비율은 0~1 사이의 값으로 나타내시오.
3. 길이 600m인 도로를 30km/h로 주행할 때의 통행시간을 초 단위로 구하시오.
   코드에는 계산식을 작성하시오.
4. 자동차용 도로만 선택해야 하는 이유를 설명하시오.
   A→B의 통행시간을 B→A에도 적용할 수 있는지 도로의 방향성과 관련지어 설명하시오.
5. `load_road_graph`가 결측 속도를 처리하는 방법을 설명하시오.
   이 방법이 혼잡 시간의 통행시간을 실제보다 짧게 계산할 수 있는 이유를 서술하시오.

1~3번은 아래 코드에, 4~5번은 답안 셀에 작성하시오.

참고: 2장, 4.2~4.4절. 결측 비율은 `edges["컬럼"].isna().mean()`으로 구할 수 있습니다.

In [ ]:
# TODO: None을 계산식으로 바꿉니다.
node_count = None
edge_count = None
missing_share = None
travel_seconds = None  # 600m, 30km/h인 도로 한 구간의 통행시간(초)

# 확인 코드는 그대로 둡니다.
expect("자동차 노드 수", node_count, g_free.n_nodes)
expect("자동차 엣지 수", edge_count, g_free.n_edges)
expect("오후 첨두 속도 결측 비율", missing_share,
       edges["weekday_pm_peak_p50"].isna().mean(), tol=1e-9)
expect("600m를 30km/h로 이동하는 시간(초)", travel_seconds, 72.0, tol=1e-9)

### 문제 1 서술 답안

4번: 자동차용 도로 선택과 도로의 방향성 (2~3문장)

답:

5번: 결측 속도 처리와 통행시간 추정 (2~3문장)

답:

## 문제 2. 다익스트라 알고리즘의 탐색 과정 (25점)

다음은 노드 A, B, C, D로 구성된 방향 그래프의 엣지 목록입니다.
각 엣지의 비용은 통행시간이며 단위는 초입니다.
표에 없는 엣지와 역방향 엣지는 존재하지 않습니다.
이 그래프는 하남시 도로망과 별개로 주어진 계산 문제입니다.

| 출발 노드 | 도착 노드 | 통행시간(초) |
|---|---|---|
| A | B | 60 |
| A | C | 150 |
| B | C | 80 |
| B | D | 20 |
| D | C | 30 |

1. 다익스트라 알고리즘으로 A에서 C까지 탐색하는 과정을 아래 표에 작성하시오.
   초기 비용은 A가 0이고 나머지 노드는 무한대입니다.
   각 행에는 노드 하나를 확정하고 이웃의 비용을 갱신한 직후의 상태를 기록하시오.
   C가 확정되면 탐색을 종료하시오.
2. A에서 C까지의 최소 통행시간과 그 경로를 구하여 아래 코드에 입력하시오.
3. C가 처음 후보 목록에 추가되는 순간 탐색을 종료하면 반환되는 통행시간을 구하시오.
   이 값이 2번의 결과와 다른 이유를 비용 갱신 과정에 근거하여 설명하시오.

참고: 3.2~3.4절.

### 문제 2 답안

1번: 탐색 과정

| 단계 | 이번에 확정한 노드 | B의 현재 비용(초) | C의 현재 비용(초) | D의 현재 비용(초) | C의 직전 노드 |
|---|---|---|---|---|---|
| 초기 | 없음 | ∞ | ∞ | ∞ | 없음 |
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |

3번: C를 처음 발견했을 때 종료하는 경우 (2~3문장)

답:

2번의 경로와 통행시간은 아래 코드에 작성하시오.

In [ ]:
# TODO: 손으로 구한 경로와 최소 통행시간을 적습니다.
toy_path = None     # 예시 형식: ["A", "B", "C"]
toy_seconds = None  # 초 단위 숫자

# 경로의 연결과 비용 합을 확인합니다. 최적성은 단계별 표와 함께 설명합니다.
toy_costs = {("A", "B"): 60, ("A", "C"): 150,
             ("B", "C"): 80, ("B", "D"): 20, ("D", "C"): 30}
if toy_path is None or toy_seconds is None:
    print("[ ] 경로와 시간을 채운 뒤 다시 실행합니다.")
else:
    valid = (
        len(toy_path) >= 2 and toy_path[0] == "A" and toy_path[-1] == "C"
        and all((u, v) in toy_costs for u, v in zip(toy_path, toy_path[1:]))
    )
    expect("방향을 지키는 A→C 경로", valid, True)
    if valid:
        cost_sum = sum(toy_costs[u, v] for u, v in zip(toy_path, toy_path[1:]))
        expect("경로의 비용 합(초)", toy_seconds, cost_sum)
    print("비용 합이 맞아도 최단경로라는 뜻은 아닙니다. 단계별 표로 확인합니다.")

## 문제 3. 자유류와 오후 첨두의 경로·통행시간 비교 (35점)

다음 세 구간에 대해 자유류 속도와 주중 오후 첨두 속도를 각각 적용하여
최소 통행시간과 경로를 구하시오.

| 구간 | 출발지 | 목적지 |
|---|---|---|
| A→B | 하남시청 | 미사역 |
| B→A | 미사역 | 하남시청 |
| C→B | 가상 승하차 지점 C | 미사역 |

지점별 좌표와 도로망 노드는 앞에서 제공한 `places`와 `node_ids`를 사용하시오.
자유류 계산에는 `g_free`, 오후 첨두 계산에는 `g_peak`를 사용하시오.
각 구간의 출발·도착 노드는 두 계산에서 동일하게 유지하시오.
두 그래프에서 다익스트라를 각각 실행하여 총 여섯 개의 경로를 구하시오.

참고: 3장 실습, 4.3절.

### 3-1. 경로와 통행시간 계산

`route_one(graph, source, target)`의 빈칸을 완성하시오.
이 함수는 주어진 도로망과 출발·도착 노드에 대해 다익스트라를 실행하고,
경로 객체와 분 단위 통행시간을 반환해야 합니다.
완성한 함수를 이용하여 세 구간의 자유류 통행시간, 오후 첨두 통행시간, 경로 변경 여부를 표로 제시하시오.

In [ ]:
def route_one(graph, source, target):
    # TODO 1: 위의 함수 설명 표를 보고 다익스트라를 호출합니다.
    path = None
    if path is None:
        return None

    # TODO 2: 초 단위 통행시간을 분으로 바꿉니다.
    minutes = None
    if minutes is None:
        return None

    return path, minutes

아래 코드는 `route_one`을 호출하여 구간별 결과를 표로 정리합니다.
`경로_변경`은 자유류와 오후 첨두의 경로 노드 목록이 다르면 `True`, 같으면 `False`입니다.

In [ ]:
paths = {}
rows = []
comparison = None

for _, trip in trips.iterrows():
    trip_id = trip["구간"]
    row = {"구간": trip_id}
    for label, graph in graphs.items():
        result = route_one(graph, node_ids[trip["출발"]], node_ids[trip["도착"]])
        if result is None:
            continue
        path, minutes = result
        paths[(trip_id, label)] = path
        row[f"{label}_분"] = minutes
    if "자유류_분" in row and "오후 첨두_분" in row:
        row["경로_변경"] = paths[(trip_id, "자유류")].nodes != paths[(trip_id, "오후 첨두")].nodes
        rows.append(row)

if len(rows) == len(trips):
    comparison = pd.DataFrame(rows).set_index("구간")
    display(comparison.round(2))
else:
    print("[ ] route_one의 빈칸을 채운 뒤 이 셀을 다시 실행합니다.")

### 3-2. 통행시간 증가량과 증가율 계산

각 구간에서 오후 첨두 통행시간이 자유류 통행시간보다 얼마나 증가했는지 계산하시오.
증가량은 분, 증가율은 %로 나타내시오.

$$
\text{증가량} = \text{오후 첨두 통행시간} - \text{자유류 통행시간}
$$

$$
\text{증가율(\%)} =
\frac{\text{오후 첨두 통행시간} - \text{자유류 통행시간}}{\text{자유류 통행시간}} \times 100
$$

세 구간의 통행시간 증가량에 대해 평균과 최댓값을 구하시오.
계산 중에는 반올림하지 않고, 결과를 출력할 때 소수 둘째 자리까지 표시하시오.

참고: `comparison["자유류_분"]`은 자유류 통행시간 컬럼입니다.
컬럼의 평균은 `.mean()`, 최댓값은 `.max()`로 구할 수 있습니다.

In [ ]:
mean_increase = None
max_increase = None

if comparison is None:
    print("[ ] 3-1의 결과를 먼저 만듭니다.")
else:
    # TODO: None을 컬럼 사이의 계산식으로 바꿉니다.
    comparison["증가_분"] = None
    comparison["증가율_%"] = None

    if comparison[["증가_분", "증가율_%"]].isna().any().any():
        print("[ ] 증가 시간과 증가율을 채웁니다.")
    else:
        # TODO: 증가 시간의 평균과 최댓값을 구합니다.
        mean_increase = None
        max_increase = None
        display(comparison.round(2))
        todo("세 구간의 평균 증가 시간(분)", mean_increase)
        todo("가장 큰 증가 시간(분)", max_increase)

### 3-3. 경로 비교와 결과 해석

제공된 그림 코드를 이용하여 A→B의 자유류 경로와 오후 첨두 경로를 하나의 그림에 나타내시오.
구간별 비교표와 그림을 근거로 아래 서술 문항에 답하시오.

그림의 선은 경로 노드를 순서대로 연결한 것입니다.
배경 지도와 도로의 세밀한 곡선은 생략되어 있습니다.

In [ ]:
if comparison is None:
    print("[ ] 3-1의 결과를 먼저 만듭니다.")
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    for label, style in [("자유류", "-"), ("오후 첨두", "--")]:
        coords = paths[("A→B", label)].coords(graphs[label])
        positions = pd.DataFrame(coords, columns=["위도", "경도"])
        ax.plot(positions["경도"], positions["위도"], style, label=label)
    for key in ["A", "B"]:
        lat, lon = g_free.coord[node_ids[key]]
        ax.scatter(lon, lat, color="black", zorder=3)
        ax.annotate(f"{key}: {places.loc[key, '이름']}", (lon, lat),
                    xytext=(5, 5), textcoords="offset points")
    ax.set_xlabel("경도")
    ax.set_ylabel("위도")
    ax.set_title("A→B: 자유류와 오후 첨두의 경로")
    ax.legend()
    fig.tight_layout()
    plt.show()

### 문제 3 서술 답안

1. 통행시간이 가장 많이 증가한 구간과 증가량을 제시하시오.
   이 값을 세 구간의 평균 증가량과 비교하고, 평균만으로 설명할 때 놓칠 수 있는 차이를 서술하시오. (2~3문장)
   소수 둘째 자리까지 표시한 증가량이 같으면 해당 구간을 모두 제시해도 됩니다.

   답:

2. 자유류와 오후 첨두에서 각각 A→B와 B→A의 통행시간을 비교하시오.
   A→B의 경로가 자유류와 오후 첨두에서 달라지는지도 그림을 근거로 서술하시오. (2~3문장)

   답:

3. 모든 엣지의 속도가 기존 속도의 절반으로 감소했다고 가정하시오.
   각 경로의 통행시간은 몇 배가 되는지, 최소 통행시간 경로가 달라지는지 설명하시오.
   이를 도로마다 속도 감소 비율이 다른 경우와 비교하시오. (2~3문장)

   답:

## 문제 4. 차량의 다음 배차 가능 시각 (20점)

택시 한 대가 18:00에 호출을 받아 하남시청(A)에서 승객을 태운 뒤 미사역(B)으로 이동합니다.
운행 단계별 소요시간은 다음과 같습니다.

| 단계 | 소요시간 |
|---|---|
| 차량의 현재 위치에서 A까지 이동 | 4분 |
| A에서 승차 | 1분 |
| A→B 이동 | 문제 3에서 구한 자유류 또는 오후 첨두 통행시간 |
| B에서 하차 | 1분 |

A→B 통행시간에 자유류 계산값을 적용한 경우와 오후 첨두 계산값을 적용한 경우를 비교하시오.
A까지 이동하는 시간과 승하차 시간은 두 경우에 동일하게 적용하시오.
차량은 하차가 완료된 순간부터 다음 호출을 받을 수 있다고 가정합니다.

1. 각 경우에 호출부터 하차 완료까지 걸리는 시간을 분 단위로 구하시오.
   이를 이용하여 다음 배차가 가능한 시각을 구하시오.
2. 각 경우에 18:13에 들어온 새 호출을 즉시 받을 수 있는지 판정하시오.
   일정표의 `18:13_배차가능` 컬럼에 `True` 또는 `False`로 나타내시오.
3. 아래 서술 문항에 답하시오.

참고: 1.2~1.3절, 4.1절.
1번의 소요시간은 코드에, 배차 가능 시각은 서술 답안에 작성하시오.

In [ ]:
schedule = None
if comparison is None:
    print("[ ] 문제 3의 통행시간을 먼저 계산합니다.")
else:
    schedule = pd.DataFrame({
        "A→B_분": [comparison.loc["A→B", "자유류_분"],
                   comparison.loc["A→B", "오후 첨두_분"]],
    }, index=["자유류", "오후 첨두"])
    pickup_min = 4
    boarding_min = 1
    alighting_min = 1

    # TODO: 픽업, 승차, A→B 이동, 하차 시간을 더합니다.
    schedule["하차완료까지_분"] = None
    if schedule["하차완료까지_분"].isna().any():
        print("[ ] 하차 완료까지 걸리는 시간을 채웁니다.")
    else:
        # TODO: 하차를 마쳤는지 판정하는 비교식을 적습니다.
        schedule["18:13_배차가능"] = None
        display(schedule.round(2))

### 문제 4 서술 답안

다음 항목에 대해 총 6~8문장으로 답하시오.
배차 가능 시각은 초 단위를 반올림하여 시:분:초로 나타내시오.

- 자유류 계산값과 오후 첨두 계산값을 적용했을 때 다음 배차가 가능한 시각을 각각 제시하시오.
  18:13의 새 호출에 대한 판정 결과도 함께 제시하시오.
- 두 경우의 차이가 다음 승객의 대기와 차량 운영에 미칠 수 있는 영향을 설명하시오.
- 이 비교에서 동일하게 유지한 조건 두 가지와 변경한 조건 한 가지를 제시하시오.
- 이 결과만으로 하남시 전체 승객의 대기시간 증가량을 추정할 수 있는지 설명하시오.
  결측 속도 대체, 좌표와 도로망 노드의 간격, 분석 대상 세 구간의 대표성 중 하나를 근거로 포함하시오.

답:

## 제출 안내

코드, 출력 표, 경로 그림, 서술 답안을 이 노트북에 작성하시오.
파일 이름을 `assignment_1_학번.ipynb`로 변경하고, 실행 결과가 저장된 상태로 제출하시오.
별도 보고서는 제출하지 않습니다.

제출 전 커널을 다시 시작하고 모든 셀을 위에서부터 실행하여 다음 항목을 확인하시오.

- [ ] 문제 1의 계산 결과와 서술 답안을 작성했습니다.
- [ ] 문제 2의 탐색 과정 표, 최단경로, 통행시간과 조기 종료에 관한 답안을 작성했습니다.
- [ ] 문제 3의 비교표, 평균·최대 증가량, 경로 그림과 서술 답안을 작성했습니다.
- [ ] 문제 4의 차량 일정표, 배차 가능 시각과 서술 답안을 작성했습니다.
- [ ] 계산에 사용한 단위를 명시하고, 모형의 계산 결과와 실제 관측값을 구분했습니다.

문제별 배점은 첫 표와 같습니다.
코드의 실행 여부, 계산의 정확성, 결과 해석의 근거를 함께 평가합니다.
제공된 확인 코드는 일부 계산과 형식만 확인하며, 서술 답안은 별도로 평가합니다.